# Tutorial: Checkpointer Explorer Showcase

Audience:
- Hypergraph users who want one place to inspect persisted workflow state, nested runs, steps, checkpoints, and lineage.

Prerequisites:
- Basic familiarity with `Graph`, `AsyncRunner`, and `SqliteCheckpointer`.
- `aiosqlite` installed in the active environment.

Learning goals:
- Build a small run history with completed, failed, retried, forked, and nested workflows.
- Use the new checkpointer explorer as the top-level drill-through UI.
- Jump from checkpointer to runs, steps, checkpoints, and lineage from one notebook.


## Outline

1. Setup a fresh demo database
2. Create a small but varied workflow history
3. Open the unified checkpointer explorer
4. Inspect runs, steps, checkpoints, and lineage directly
5. Clean up the database connection


In [ ]:
from __future__ import annotations

from pathlib import Path

from hypergraph import AsyncRunner, Graph, interrupt, node
from hypergraph.checkpointers import SqliteCheckpointer

NOTEBOOK_DIR = Path("output/jupyter-notebook")
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = NOTEBOOK_DIR / "checkpointer-explorer-showcase.db"
if DB_PATH.exists():
    DB_PATH.unlink()

DB_PATH

## Step 1 - Build a mixed run history

The next cell creates:
- a normal completed workflow
- a forked workflow
- a failed workflow plus retry lineage
- a nested interrupt workflow that is resumed through a fork

This gives the explorer enough interesting structure to drill through.

In [ ]:
@node(output_name="doubled")
def double(x: int) -> int:
    return x * 2


@node(output_name="tripled")
def triple(doubled: int) -> int:
    return doubled * 3


@node(output_name="draft")
def make_draft(query: str) -> str:
    return f"Draft for: {query}"


@interrupt(output_name="decision")
def approval(draft: str) -> str: ...


@node(output_name="final")
def finalize(decision: str) -> str:
    return f"Final: {decision}"


basic_graph = Graph([double, triple], name="basic")
review_graph = Graph([approval], name="review")
nested_graph = Graph([make_draft, review_graph.as_node(name="review"), finalize], name="nested-review")


async def build_demo() -> dict[str, object]:
    checkpointer = SqliteCheckpointer(str(DB_PATH), durability="sync")
    runner = AsyncRunner(checkpointer=checkpointer)

    base = await runner.run(basic_graph, {"x": 5}, workflow_id="demo-basic")
    base_checkpoint = checkpointer.checkpoint("demo-basic")
    forked = await runner.run(basic_graph, {"x": 7}, checkpoint=base_checkpoint, workflow_id="demo-basic-fork")

    fail_once = {"value": True}

    @node(output_name="seed")
    def seed(x: int) -> int:
        return x

    @node(output_name="out")
    def flaky(seed: int) -> int:
        if fail_once["value"]:
            fail_once["value"] = False
            raise RuntimeError("transient failure")
        return seed * 10

    flaky_graph = Graph([seed, flaky], name="flaky")
    failed = await runner.run(flaky_graph, {"x": 3}, workflow_id="demo-retry-root", error_handling="continue")
    retried = await runner.run(flaky_graph.with_entrypoint("flaky"), retry_from="demo-retry-root", workflow_id="demo-retry-root-1")

    paused = await runner.run(nested_graph, {"query": "hello reviewer"}, workflow_id="demo-nested")
    resumed_fork = await runner.run(
        nested_graph,
        {paused.pause.response_key: "approved"},
        fork_from="demo-nested",
        workflow_id="demo-nested-fork",
    )

    return {
        "checkpointer": checkpointer,
        "base": base,
        "forked": forked,
        "failed": failed,
        "retried": retried,
        "paused": paused,
        "resumed_fork": resumed_fork,
    }

In [ ]:
demo = await build_demo()

In [ ]:
sorted(demo.keys())

## Step 2 - Open the unified explorer

This is the new top-level notebook UI.

Start here, then click through runs, child runs, source lineage, and steps from a single surface.

In [ ]:
cp = demo["checkpointer"]
cp

## Step 3 - The lower-level views still work

The explorer is the new entry point, but the individual views remain useful when you want a focused object.

In [ ]:
cp.runs(limit=20)

In [ ]:
cp.steps("demo-nested-fork/review")

In [ ]:
cp.checkpoint("demo-basic")

In [ ]:
cp.lineage("demo-nested-fork", include_steps=True)

## Step 4 - A few direct lookups

These are handy when you already know the run ID and want a precise answer rather than browsing.

In [ ]:
{
    "resumed_nested_final": demo["resumed_fork"]["final"],
    "retry_run_status": demo["retried"].status.value,
    "known_child_runs": [run.id for run in cp.runs(limit=20) if run.parent_run_id],
}